# 🏢 AmbitionBox Company Scraper — Detailed Notes

This notebook scrapes company data (name, rating, reviews, pros, cons) from [AmbitionBox](https://www.ambitionbox.com/list-of-companies) using Python web-scraping tools.

---

## Libraries Used
| Library | Purpose |
|---------|---------|
| `pandas` | Create and manipulate tabular DataFrames |
| `requests` | Send HTTP requests to fetch webpage HTML |
| `BeautifulSoup` | Parse and navigate the HTML DOM tree |
| `time` | Add delays between requests to avoid getting blocked |

---

## 📦 Cell 1 — Importing Libraries

### What this does:
- **`pandas`** — used at the end to create a structured DataFrame from scraped lists.
- **`requests`** — sends a GET request to the AmbitionBox URL and returns the raw HTML of the page.
- **`BeautifulSoup`** — parses that raw HTML string into a navigable tree, so we can search for specific tags, classes, and text.

### Key concept:
`from bs4 import BeautifulSoup` — `bs4` is the package name (Beautiful Soup version 4), and `BeautifulSoup` is the main class inside it.

In [1]:
import pandas as pd
import requests
from bs4 import  BeautifulSoup

---
## 🌐 Cell 2 — Sending an HTTP GET Request

### What this does:
1. **`headers`** — A dictionary sent along with the HTTP request. Websites block automated bots that don't look like a real browser. By setting `User-Agent` to a real browser string, we "disguise" our request as coming from a Chrome browser on Windows.

2. **`requests.get(url, headers=headers)`** — Makes a GET request to the AmbitionBox page 1 URL. Returns a `Response` object.

3. **`.text`** — Extracts the raw HTML string from the response (i.e., the entire source code of the webpage as a string).

### Why headers matter:
Without a proper `User-Agent`, AmbitionBox may return a `403 Forbidden` or redirect to a CAPTCHA page, giving you empty/wrong HTML.

### Variable stored:
- `webpage` → raw HTML string of the page

In [3]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko) Chrome/80.0.3987.162 Safari/537.36'}
webpage = requests.get('https://www.ambitionbox.com/list-of-companies?page=1',headers=headers).text

---
## 🍵 Cell 3 — Parsing HTML with BeautifulSoup

### What this does:
`BeautifulSoup(webpage, 'lxml')` takes the raw HTML string (`webpage`) and parses it using the `lxml` parser.

### What is a parser?
A **parser** reads the raw HTML text and converts it into a tree of objects (tags, attributes, text). The `lxml` parser is very fast and forgiving of malformed HTML.

### Available parsers:
| Parser | Speed | Notes |
|--------|-------|-------|
| `'lxml'` | Fast | Requires `pip install lxml` |
| `'html.parser'` | Medium | Built-in Python, no extra install |
| `'html5lib'` | Slow | Most lenient, parses like a browser |

### Variable stored:
- `soup` → A `BeautifulSoup` object you can search using `.find()`, `.find_all()`, `.select()`, etc.

In [4]:
soup=BeautifulSoup(webpage,'lxml')

---
## 🖨️ Cell 4 — Viewing the Parsed HTML (prettify)

### What this does:
`soup.prettify()` returns the entire HTML tree as a nicely indented string — useful for visually inspecting the structure.

### When to use:
- **Debugging** — to see what the actual HTML looks like before deciding which tags/classes to target.
- The output is very long; in practice you'd scroll through it or `print()` just a section.

> ⚠️ This cell is mostly for exploration. The real scraping happens later.

In [5]:
print(soup.prettify())

<!DOCTYPE html>\n<html ...>\n ... (full HTML output) ...\n

---
## 🔍 Cell 5 — Finding All `<h1>` Tags

### Method: `soup.find_all('h1')`

### What `find_all()` does:
- Searches the **entire HTML tree** for all tags matching the given criteria.
- Returns a **list** of `Tag` objects (even if only one match exists).
- Each `Tag` object contains the tag's text, attributes, and children.

### Difference between `find()` and `find_all()`:
| Method | Returns |
|--------|---------|
| `soup.find('h1')` | First matching tag only (or `None`) |
| `soup.find_all('h1')` | List of ALL matching tags |

### Result here:
AmbitionBox only has one `<h1>` — the "Top Companies in India" heading, which is the main page heading used for SEO.

In [7]:
soup.find_all('h1')

---
## 🔍 Cell 6 — Extracting Text from All `<h2>` Tags

### What this does:
Loops over every `<h2>` tag on the page and prints its text content.

### Key methods used:
- **`.text`** — Returns all text inside a tag, including text in nested child tags, concatenated together.
- **`.strip()`** — Removes leading and trailing whitespace/newlines from the string.

### Observation from output:
The `<h2>` tags contain:
- Company names (e.g., "TCS", "Accenture", "Wipro" …)
- Section headings like "Popular Collections by Industries", "Popular Collections by Cities", etc.

This confirms that company names are wrapped in `<h2>` tags with the class `companyCardWrapper__companyName`.

In [8]:
for i in soup.find_all('h2'):
  print(i.text.strip())

Companies in India
TCS
Accenture
...etc


---
## 🔍 Cell 7 — Extracting Text from All `<p>` Tags

### What this does:
Loops over every `<p>` (paragraph) tag and prints its stripped text.

### Purpose:
Exploring what `<p>` tags contain — useful for finding descriptions, taglines, or other metadata. From the output we can see `<p>` tags contain things like:
- "AmbitionBox"
- Navigation descriptions like "Discover best places to work"
- The copyright line at the bottom

This exploration helps decide which tags to actually target for data extraction.

In [9]:
for i in soup.find_all('p'):
  print(i.text.strip())

AmbitionBox
About Company
...etc


---
## 🔢 Cell 8 — Counting Rating Text Divs

### What this does:
`soup.find_all('div', class_='rating_text')` finds all `<div>` elements with **exactly** the class `rating_text`.

`len(...)` counts how many are found → **20**, confirming one rating div per company (20 companies on page 1).

### Why count?
Counting is a sanity check — if there are 20 companies and 20 rating divs, the selector is correct. If the count was 0, the class name would be wrong.

### Class-based selection:
`soup.find_all('tag', class_='some-class')` is one of the most common BeautifulSoup patterns. The CSS class name is taken directly from the browser's "Inspect Element" tool.

In [12]:
len(soup.find_all('div',class_='rating_text'))

20

---
## 🔢 Cell 9 — Counting Action Wrapper Links

### What this does:
`soup.find_all('a', class_='companyCardWrapper__ActionWrapper')` finds all anchor (`<a>`) tags with the given class.

Result: **120** links → 6 action links per company × 20 companies = 120. ✅

### The 6 action links per company are:
1. Reviews
2. Salaries
3. Interviews
4. Jobs
5. Benefits
6. Photos

### Key insight:
This tells us that `companyCardWrapper__ActionWrapper` links contain the **count data** (e.g., "1.1L Reviews", "10L Salaries"). We'll extract just the first one (Reviews count) per company later.

In [15]:
len(soup.find_all('a', class_='companyCardWrapper__ActionWrapper'))

120

---
## 🏢 Cell 10 — Selecting All Company Card Containers

### What this does:
`soup.find_all('div', class_='companyCardWrapper')` selects every company card on the page.

Each `<div class="companyCardWrapper">` is a self-contained block that holds **all data for one company** — name, rating, reviews, pros, cons, action counts, etc.

### Why store in a variable?
By storing all cards in `company`, we can then **loop over each card** and extract specific data fields from it, instead of searching the entire page each time.

### Result:
`company` is a Python **list** of 20 BeautifulSoup `Tag` objects, one per company card.

In [48]:
company=soup.find_all('div',class_='companyCardWrapper')

---
## 👀 Cell 11 — Inspecting the `company` List

### What this does:
Simply evaluates `company` (the list of 20 company card tags) in the notebook, which displays its full content.

### Purpose:
Before writing extraction code, it's good practice to **visually inspect** what you've captured. You can see each tag's full HTML, which lets you identify:
- The exact class names to target
- Nesting structure of elements
- Which tags contain the data you need

> 💡 In real projects, you'd inspect just one card: `company[0]` to study the structure.

In [49]:
company

[<div class=\"companyCardWrapper\" ...>...</div>, ...]

---
## 🔄 Cell 12 — Main Data Extraction Loop (Single Page)

### Overview:
This is the core scraping logic. We loop over each company card and extract 5 fields.

### Data structures used:
Five empty **Python lists** are created first. Each iteration of the loop appends one value to each list, so all lists end up the same length (20).

### Field-by-field breakdown:

#### 1. Company Name
```python
i.find('h2', class_='companyCardWrapper__companyName').text.strip()
```
- Finds the `<h2>` tag with that class inside the current card `i`.
- `.text` extracts all text content. `.strip()` removes whitespace.

#### 2. Rating
```python
i.find('div', class_='rating_text').text.strip()
```
- The star rating (e.g., "3.3") lives inside a `<div class="rating_text">`.

#### 3. Review Count (first ActionCount)
```python
i.find('span', class_='companyCardWrapper__ActionCount').text.strip()
```
- `.find()` (not `find_all`) returns only the **first** matching span — which is the Reviews count.

#### 4 & 5. Positives and Negatives
```python
tags = i.find_all('span', class_='companyCardWrapper__ratingValues')
positive.append(tags[0].get_text(strip=True))  # first span = positives
negative.append(tags[1].get_text(strip=True))  # second span = negatives
```
- There are at most 2 `ratingValues` spans per card: one for "Highly Rated For" and one for "Critically Rated For".
- We guard with `if len(tags) >= 1` and `if len(tags) >= 2` to handle cards that have only one or neither.
- `.get_text(strip=True)` is similar to `.text.strip()`.

### Finally:
`pd.DataFrame({...})` assembles all 5 lists into a structured table with named columns.

In [76]:
name=[]
rating=[]
reviews=[]
positive=[]
negative=[]


for i in company:
    name.append(i.find('h2',class_='companyCardWrapper__companyName').text.strip())
    
    rating.append(i.find('div',class_='rating_text').text.strip())
    
    reviews.append(i.find('span', class_='companyCardWrapper__ActionCount').text.strip())

    tags = i.find_all('span', class_='companyCardWrapper__ratingValues')
    
    if len(tags) >= 1:
        positive.append(tags[0].get_text(strip=True))
    else:
        positive.append(None)
    
    if len(tags) >= 2:
        negative.append(tags[1].get_text(strip=True))
    else:
        negative.append(None)

df=pd.DataFrame({'name':name,
   'rating':rating,
   'reviews':reviews,
   'positives':positive,
   'negatives':negative,
   })
    
    

---
## 📊 Cell 13 — Displaying the Single-Page DataFrame

### What this does:
Evaluates `df` to render it as a formatted HTML table in Jupyter Notebook.

### What to check in the output:
- **20 rows** (one per company on page 1) ✅
- All 5 columns populated ✅
- Some `None` in `negatives` for companies with no critically-rated fields — this is expected ✅

In [77]:
df

---
## 📋 Cells 14–18 — Displaying Individual Extracted Lists

These cells display `name`, `rating`, `reviews`, `positive`, and `negative` lists individually for verification.

### Purpose:
- Quick sanity check that each list has exactly **20 items** and the values look correct.
- Helps debug if one list is shorter/longer than expected (which would cause a DataFrame error).

### Key points:
- `rating` is a list of strings like `'3.3'`, `'4.6'` — not floats. To do math you'd need `float(r)`.
- `reviews` uses abbreviated notation: `'1.1L'` = 1.1 Lakh = 110,000; `'72.9k'` = 72,900.
- `None` in `negative` means that company only had a positively-rated category (no critical ratings).

In [55]:
name

In [56]:
rating

In [69]:
reviews

In [74]:
positive

In [75]:
negative

---
## 🔄 Cell 19 — Multi-Page Scraper (FIXED ✅)

### What this does:
Loops over **pages 1–19** of AmbitionBox, scrapes each page, and concatenates all results into one large DataFrame called `final`.

### Bug that was fixed:
The original code used:
```python
company = soup.find_all('div', class_='company-content-wrapper')
```
❌ `company-content-wrapper` is **NOT a real CSS class** in AmbitionBox's HTML.
This returned an empty list every time → all inner lists stayed empty → `final` was an empty DataFrame.

**Fixed to:**
```python
company = soup.find_all('div', class_='companyCardWrapper')
```
✅ This is the correct class name (verified from the single-page exploration above).

---

### Code structure explained:

#### Outer loop: `for j in range(1, 20)`
Iterates page numbers 1 through 19. Each iteration constructs a fresh URL:
```python
url = f'https://www.ambitionbox.com/list-of-companies?page={j}'
```

#### `try / except` block
Wraps each page's request in error handling. If a page times out or fails:
- The `except` clause catches the exception and prints a friendly message.
- `continue` skips to the next page instead of crashing the whole loop.

#### Status code check
```python
if response.status_code != 200:
    print(f"Skipping page {j}")
    continue
```
`200` means OK. Other codes (403 Forbidden, 429 Too Many Requests, etc.) mean something went wrong — we skip that page.

#### `timeout=10`
If the server doesn't respond within 10 seconds, `requests` raises a `Timeout` exception — caught by the `except` block.

#### `time.sleep(1)`
Pauses for 1 second between requests. This is **anti-blocking** — sending requests too fast looks like a bot attack and may get your IP banned or CAPTCHAed.

#### `all_dfs.append(df)`
Each page's DataFrame is appended to the `all_dfs` list.

#### `pd.concat(all_dfs, ignore_index=True)`
Combines all per-page DataFrames into one big DataFrame. `ignore_index=True` resets the row index to 0, 1, 2, … instead of keeping each page's 0–19 index.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_dfs = []

for j in range(1, 20):
    try:
        url = f'https://www.ambitionbox.com/list-of-companies?page={j}'
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Skipping page {j}")
            continue
        
        soup = BeautifulSoup(response.text, 'lxml')
        company = soup.find_all('div', class_='companyCardWrapper')  # ✅ FIXED class name

        name, rating, reviews, positive, negative = [], [], [], [], []

        for i in company:
            name.append(i.find('h2', class_='companyCardWrapper__companyName').text.strip())
            
            r = i.find('div', class_='rating_text')
            rating.append(r.text.strip() if r else None)
            
            rev = i.find('span', class_='companyCardWrapper__ActionCount')
            reviews.append(rev.text.strip() if rev else None)

            tags = i.find_all('span', class_='companyCardWrapper__ratingValues')
            positive.append(tags[0].get_text(strip=True) if len(tags) > 0 else None)
            negative.append(tags[1].get_text(strip=True) if len(tags) > 1 else None)

        df = pd.DataFrame({
            'name': name,
            'rating': rating,
            'reviews': reviews,
            'positives': positive,
            'negatives': negative
        })

        all_dfs.append(df)

        print(f"✅ Page {j} done")

        time.sleep(1)  # anti-block delay

    except Exception as e:
        print(f"❌ Error on page {j}: {e}")
        continue

final = pd.concat(all_dfs, ignore_index=True)

✅ Page 1 done
✅ Page 2 done
✅ Page 3 done
✅ Page 4 done
✅ Page 5 done
✅ Page 6 done
✅ Page 7 done
✅ Page 8 done
✅ Page 9 done
✅ Page 10 done
✅ Page 11 done
✅ Page 12 done
✅ Page 13 done
✅ Page 14 done
✅ Page 15 done
✅ Page 16 done
✅ Page 17 done
✅ Page 18 done
✅ Page 19 done


---
## 📊 Cell 20 — Displaying the Final Multi-Page DataFrame

### What this does:
Displays `final` — the combined DataFrame from all successfully scraped pages.

### Expected output:
With pages 1–19 (minus 1 timed-out page = 18 pages × 20 companies), you'd expect approximately **360 rows**.

### Next steps you could take:
- `final.to_csv('ambitionbox_companies.csv', index=False)` — save to CSV
- `final['rating'] = final['rating'].astype(float)` — convert ratings to float for analysis
- `final.sort_values('rating', ascending=False)` — find highest-rated companies
- `final.dropna(subset=['negatives'])` — filter only companies with critical ratings

In [2]:
final

,name,rating,reviews,positives,negatives
0,TCS,3.3,1.1L,Job Security,"Promotions, Salary, Work Satisfaction"
1,Accenture,3.7,73k,"Promotions, Salary, Work Satisfaction",None
2,Wipro,3.6,64.8k,"Promotions, Salary, Work Satisfaction",None
3,Cognizant,3.7,61.1k,"Promotions, Salary, Work Satisfaction",None
4,Capgemini,3.7,52.8k,"Work Life Balance, Job Security","Promotions, Salary, Work Satisfaction"
...,...,...,...,...,...
375,Adani Group,3.8,2.4k,Salary,Promotions
376,Intellect Design Arena,3.6,2.4k,Job Security,"Promotions, Work Satisfaction, Company Culture"
377,CorroHealth Infotech,3.6,2.4k,Salary,"Work Life Balance, Work Satisfaction, Job Secu..."
378,Datamatics Global Services,3.4,2.4k,"Promotions, Salary, Work Satisfaction",None
